In [4]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Locate project root dynamically
current_dir = Path(os.getcwd())
root_dir = current_dir
for parent in [current_dir] + list(current_dir.parents):
    if (parent / "Data sets").exists():
        root_dir = parent
        break

# File path resolution
file_path = root_dir / "Data sets" / "3) House Price Prediction.csv"
if not file_path.exists():
    data_dir = root_dir / "Data sets"
    file_path = next((f for f in data_dir.glob("*.csv") if "house" in f.name.lower() or "3)" in f.name), None)

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

print(f"Loaded: {file_path.name}")
print(f"Dataset Shape: {df.shape}")

# Define target variable and feature set
target_col = next((col for col in df.columns if 'price' in col.lower() or 'target' in col.lower()), df.columns[-1])
X = df.drop(columns=[target_col])
y = df[target_col]

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Target Column: '{target_col}'")
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")

Loaded: 3) Sentiment dataset.csv
Dataset Shape: (732, 15)
Target Column: 'Hour'
Train samples: 585 | Test samples: 147


In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Feature segregation
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Define numerical and categorical transformers
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Build model pipelines
lr_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
rf_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

# Train both models
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

print("Linear Regression & Random Forest models trained successfully!")

C:\Users\manul\AppData\Local\Temp\ipykernel_23432\3624636469.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


Linear Regression & Random Forest models trained successfully!


In [6]:
from sklearn.metrics import mean_squared_error, r2_score

# Generate predictions
lr_preds = lr_pipeline.predict(X_test)
rf_preds = rf_pipeline.predict(X_test)

# Calculate RMSE and R2 metrics
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

# Summary table
metrics_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'RMSE ($)': [f"${lr_rmse:,.2f}", f"${rf_rmse:,.2f}"],
    'R² Score': [f"{lr_r2:.4f}", f"{rf_r2:.4f}"]
})

print("=== TASK 1 MODEL PERFORMANCE COMPARISON ===")
print(metrics_df.to_string(index=False))

=== TASK 1 MODEL PERFORMANCE COMPARISON ===
                  Model RMSE ($) R² Score
      Linear Regression    $3.57   0.2884
Random Forest Regressor    $3.90   0.1495
